In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from nptdms import TdmsFile
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager

# 한글 폰트 설정 (예: 나눔고딕)
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

# TDMS 파일 하나를 vib/op 사전으로 읽어오는 함수
def load_tdms_file(fp):
    tdms = TdmsFile.read(fp)
    g_vib, g_op = tdms.groups()[0].name, tdms.groups()[1].name
    vib = {f"vib_{ch.name.strip()}": ch.data for ch in tdms[g_vib].channels()}
    op  = {f"op_{ch.name.strip()}":  ch.data for ch in tdms[g_op].channels()}
    return vib, op

# 폴더 단위로 읽어서 merge한 DataFrame 반환
def load_and_merge_tdms(folder, join_how="outer"):
    dfs = []
    for fp in sorted(glob.glob(os.path.join(folder, "*.tdms"))):
        vib, op = load_tdms_file(fp)
        vib_df = pd.DataFrame(vib)
        op_df  = pd.DataFrame(op)
        merged = pd.concat([vib_df, op_df], axis=1, join=join_how)
        merged["source_file"] = os.path.basename(fp)
        dfs.append(merged)
    return pd.concat(dfs, ignore_index=True)

# 사이클 번호를 붙이는 함수
def assign_cycles(df, fs=25600, window_s=10):
    samples_per_cycle = fs * window_s
    n_cycles = len(df) // samples_per_cycle
    df = df.iloc[: n_cycles * samples_per_cycle].copy()
    df["cycle"] = np.repeat(np.arange(n_cycles), samples_per_cycle)
    return df

# Train1~8 폴더 경로 리스트 생성
base_folder = r'/Users/qwergs/Desktop/기계데이터분석(베어링)/TrainSet'
folders = [os.path.join(base_folder, f'Train{i}') for i in range(1, 9)]

# 각 폴더를 읽어서 df1~df8에 할당
dfs = []
for path in folders:
    full = load_and_merge_tdms(path, join_how="outer")
    # 진동/운전 컬럼 분리
    vib_cols = [c for c in full.columns if c.startswith("vib_")]
    op_cols  = [c for c in full.columns if c.startswith("op_")]
    vib_df = full[vib_cols].copy()
    op_df  = full[op_cols].copy()
    # cycle 번호 부여
    vib_df = assign_cycles(vib_df)
    op_df  = assign_cycles(op_df)
    # 합쳐서 하나의 DataFrame으로 사용하고 싶다면:
    merged = pd.concat([vib_df, op_df], axis=1)
    dfs.append(merged)

# 리스트 언패킹으로 df1...df8 변수에 할당
df1, df2, df3, df4, df5, df6, df7, df8 = dfs

# 확인
for i, df in enumerate(dfs, start=1):
    print(f"df{i}.shape: {df.shape}")

In [ ]:
df1.info()

- 데이터 준비 

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

WIN = 2048          # 윈도 길이(샘플)
STRIDE = WIN // 2   # 50 % 중첩
CHS = [c for c in df1.columns if c.startswith("vib_")]

def sliding_windows(arr, win=WIN, stride=STRIDE):
    n = (len(arr)-win)//stride + 1
    return np.stack([arr[i*stride: i*stride+win] for i in range(n)])

class VibDataset(Dataset):
    def __init__(self, df):
        X = df[CHS].astype(np.float32).values
        ws = [sliding_windows(X[:, i]) for i in range(X.shape[1])]
        self.data = np.stack(ws, axis=1)  # (sample, channel, win)
    def __len__(self):  return len(self.data)
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])

train_ds = VibDataset(df1)   # 정상 구간 위주 데이터
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)

- 10CNN 오토인코더

In [ ]:
import torch.nn as nn

class ConvAE(nn.Module):
    def __init__(self, in_ch=7, z_dim=64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, 32, 7, 2, 3), nn.ReLU(),
            nn.Conv1d(32, 64, 5, 2, 2),    nn.ReLU(),
            nn.Conv1d(64, 128, 5, 2, 2),   nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc   = nn.Linear(128, z_dim)
        self.d_fc = nn.Linear(z_dim, 128)
        self.dec = nn.Sequential(
            nn.ConvTranspose1d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose1d(64, 32, 4, 2, 1),  nn.ReLU(),
            nn.ConvTranspose1d(32, in_ch, 4, 2, 1)
        )
    def forward(self, x):
        h = self.enc(x).squeeze(-1)
        z = self.fc(h)
        h_dec = self.d_fc(z).unsqueeze(-1)
        x_hat = self.dec(h_dec)
        return x_hat, z

- 학습 루프 

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ConvAE(in_ch=len(CHS)).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
crit  = nn.MSELoss()

EPOCHS = 100
for ep in range(EPOCHS):
    model.train(); running = 0
    for xb in train_loader:
        xb = xb.to(device)
        x_hat, _ = model(xb)
        loss = crit(x_hat, xb)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
    if (ep+1) % 10 == 0:
        print(f"Epoch {ep+1:03d} | recon loss {running/len(train_loader):.4f}")

- 이상 점수 건강 지수 

In [ ]:
@torch.no_grad()
def anomaly_score(xb):
    x_hat, _ = model(xb.to(device))
    err = ((xb - x_hat) ** 2).mean(dim=(1,2))  # per-window MSE
    return err.cpu().numpy()

# 예: df8 검증 세트
val_ds = VibDataset(df8)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
scores = np.concatenate([anomaly_score(b) for b in val_loader])

# 임계값 (정상창 95 퍼센타일)
th = np.percentile(scores, 95)
anomaly = scores > th

- 수명 추정 간단 예

In [ ]:
# 건강지수를 0–1 범위로 정규화
hi = (scores - scores.min()) / (scores.max() - scores.min())

# hi 가 0.8 넘는 시점부터 남은 윈도 개수를 RUL 로 사용
fail_idx = np.argmax(hi > 0.8)
rul = len(hi) - fail_idx
print("예상 RUL(윈도 수):", rul)